# Prompts
> Prompts for nbdec MCP

In [ ]:
#| default_exp prompts

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

from pathlib import Path
import importlib.resources
import shutil
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import Prompt

from nbdev_mcp.utils.config import CURRENT_PROJECT, get_config
from nbdev_mcp.utils.paths import settings_dict

## Prompt loading helpers

In [ ]:
#| export
def get_python_3_9_plus_package_file(file: str, module: str = "nbdev_mcp.prompt_templates"):
    import importlib.resources
    assets = importlib.resources.files(module)
    return (assets / file).read_text(encoding="utf-8")
    
def get_python_3_8_minus_package_file(file: str, module: str = "nbdev_mcp.prompt_templates"):
    import pkgutil
    data = pkgutil.get_data(module, file)
    if data: return data.decode("utf-8")
    raise FileNotFoundError(f"Bundled template not found: {file}")

In [ ]:
#| export
def get_bundled_template(name: str) -> str:
    """Load a template from the bundled package resources.

    Uses importlib.resources which works in both installed packages and notebooks.
    """
    try: # Python 3.9+ style
        return get_python_3_9_plus_package_file(name, module="nbdev_mcp.prompt_templates")
    except (TypeError, AttributeError): # Fallback for older Python or edge cases
        return get_python_3_8_minus_package_file(name, module="nbdev_mcp.prompt_templates")

def prompt_template_path(name: str) -> Path:
    """Locate a prompt template file (for user overrides only).

    Search order:
    1. User-specified directory via NBMCP_CONFIG.prompt_dir (if absolute and exists)

    For bundled templates, use render_prompt() directly which uses importlib.resources.

    Args:
        name: Template filename (e.g., 'workflow_philosophy.md').

    Returns:
        Path to the template file.

    Raises:
        FileNotFoundError: If template cannot be found in user directory.
    """
    cfg = get_config()
    user_dir = Path(cfg.get("prompt_dir", ""))

    # Check user-specified absolute path for overrides
    if user_dir and user_dir.is_absolute() and user_dir.exists():
        candidate = user_dir / name
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"User template not found: {name}")


def prompt_context() -> Dict[str, Any]:
    """Compute default formatting context for prompts.
    
    Returns a dict with keys:
    - lib: The library name from settings.ini (or '<lib_name>' placeholder)
    - nbs_path: The notebooks directory (default 'nbs')
    """
    p = CURRENT_PROJECT
    ctx: Dict[str, Any] = {"lib": "<lib_name>", "nbs_path": "nbs"}
    if p and Path(p).exists():
        s = settings_dict(Path(p))
        ctx["lib"] = s.get("lib_name") or ctx["lib"]
        ctx["nbs_path"] = s.get("nbs_path") or ctx["nbs_path"]
    return ctx


def render_prompt(name: str, **kwargs: Any) -> str:
    """Load and format a markdown prompt template.

    Search order:
    1. User-specified directory via config (for overrides)
    2. Bundled package templates via importlib.resources

    Args:
        name: Template filename.
        **kwargs: Additional context variables for string formatting.

    Returns:
        Rendered template content.
    """
    # Try user override first
    try:
        path = prompt_template_path(name)
        raw = path.read_text(encoding="utf-8")
    except FileNotFoundError:
        # Fall back to bundled templates
        raw = get_bundled_template(name)

    ctx = prompt_context()
    ctx.update(kwargs)
    # Use safe formatting - don't fail on missing keys or complex expressions
    try:
        return raw.format(**ctx)
    except (KeyError, ValueError, AttributeError):
        # Template has complex formatting or code blocks with braces - return as-is
        return raw

## Prompts

In [ ]:
#| export
def nbdev_workflow_philosophy() -> str:
    """Foundational nbdev workflow philosophy."""
    return render_prompt("workflow_philosophy.md")


In [ ]:
#| export
def nbdev_principles() -> str:
    """Concise nbdev principles."""
    return render_prompt("nbdev_principles.md")


In [ ]:
#| export
def documentation_best_practices() -> str:
    """Documentation best practices."""
    return render_prompt("documentation_best_practices.md")


In [ ]:
#| export
def future_imports_guidance() -> str:
    """Guidance for __future__ imports."""
    return render_prompt("future_imports_guidance.md")


In [ ]:
#| export
def nbdev_howto() -> str:
    """Quick how-to for nbdev."""
    return render_prompt("nbdev_howto.md")


In [ ]:
#| export
def nbdev_documentation_guide() -> str:
    """Full nbdev documentation guide."""
    return render_prompt("documentation_guideline.md")


In [ ]:
#| export
def module_scaffold(module: str, pkg: Optional[str] = None) -> str:
    """Scaffold for new module notebooks."""
    ctx = prompt_context()
    if pkg:
        ctx["lib"] = pkg
    ctx.update({"module": module})
    return render_prompt("module_scaffold.md", **ctx)


In [ ]:
#| export
def nbdev_advanced_patterns() -> str:
    """Advanced nbdev patterns."""
    return render_prompt("advanced_patterns.md")


In [ ]:
print(nbdev_advanced_patterns()[:200])

# nbdev Advanced Patterns

## Exports
- 🚫 Do not define `__all__` anywhere. nbdev builds exports automatically.

## Package Structure
- Top-level init: `nbs/00__init__.ipynb` → `#| default_exp _init__


In [ ]:
#| export
def nbdev_main_patterns() -> str:
    """Safe __main__ patterns and console scripts."""
    return render_prompt("main_pattern.md")


In [ ]:
print(nbdev_main_patterns()[:200])

# nbdev __main__ and Console Scripts

## Safe __main__ patterns
1) Same notebook, separate cell:
```python
#| export
#| eval: False
if __name__ == "__main__":
    import sys
    sys.exit(main())
```




In [ ]:
#| export
def add_prompts(mcp: FastMCP) -> None:
    """Attach custom prompts (nbdev usage help, module scaffold) to the MCP."""
    mcp.add_prompt(Prompt(
        name="nbdev_workflow_philosophy",
        description="Foundational nbdev workflow philosophy",
        fn=nbdev_workflow_philosophy
    ))
    mcp.add_prompt(Prompt(
        name="nbdev_principles",
        description="Concise nbdev principles",
        fn=nbdev_principles
    ))
    mcp.add_prompt(Prompt(
        name="documentation_best_practices",
        description="Documentation best practices",
        fn=documentation_best_practices
    ))
    mcp.add_prompt(Prompt(
        name="future_imports_guidance",
        description="Guidance for __future__ imports",
        fn=future_imports_guidance
    ))
    mcp.add_prompt(Prompt(
        name="nbdev_howto",
        description="Quick how-to for nbdev",
        fn=nbdev_howto
    ))
    mcp.add_prompt(Prompt(
        name="nbdev_documentation_guide",
        description="Full nbdev documentation guide",
        fn=nbdev_documentation_guide
    ))
    mcp.add_prompt(Prompt(
        name="module_scaffold",
        description="Scaffold for new module notebooks",
        fn=module_scaffold
    ))
    mcp.add_prompt(Prompt(
        name="nbdev_advanced_patterns",
        description="Advanced nbdev patterns",
        fn=nbdev_advanced_patterns
    ))
    mcp.add_prompt(Prompt(
        name="nbdev_main_patterns",
        description="Safe __main__ patterns and console scripts",
        fn=nbdev_main_patterns
    ))

## Next

In [ ]:
#| export

## Export

In [ ]:
#| exporti
#| hide
def nbdev_export_templates(
    path: Path | None = None, 
    stem: str = '21_prompt_templates',
    dest: str | None = None,
):
    path = Path(path or Path()).absolute().resolve()
    path = path / stem
    for md in path.glob('*.md'):
        if dest is None: 
            dest = (md.parent / '..' / '..').absolute().resolve() / 'nbdev_mcp/prompt_templates'
        out = shutil.copy2(md, dest / md.name)

In [ ]:
#| hide
# NOTE: abusing nbdev_test // nbdev_prepare to have prompt_templates auto-exported
nbdev_export_templates()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()